In [ ]:
import torch
import torch.nn.functional as F
from gpt import MiniGPT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ---- Load tokenizer ----
import sys
from bpe_tokenizer import BPETokenizer
# Map BPETokenizer to the main namespace so torch/pickle can deserialize it
sys.modules['__main__'].BPETokenizer = BPETokenizer

vocab_data = torch.load('vocab.bin', map_location='cpu', weights_only=False)
bpe = vocab_data['bpe']

# Rebuild pre-compiled patterns if vocab.bin was saved with old tokenizer
if not hasattr(bpe, 'merge_patterns') or not bpe.merge_patterns:
    print("Upgrading vocab.bin (one-time pattern rebuild)...")
    bpe.build_vocab()

vocab_size = len(bpe.stoi)
encode = bpe.encode_ids
decode = bpe.decode_ids
print(f'Vocab size: {vocab_size}')


d_model        = 384
num_heads      = 12
num_layers     = 6
context_length = 256
dropout        = 0.1

model = MiniGPT(vocab_size, d_model, num_heads, num_layers, context_length, dropout)
model.load_state_dict(torch.load('minigpt_weights.pth', map_location=device))
model.to(device)
model.eval()
print('Model loaded successfully.')

Using device: cpu
Vocab size: 7108
Model loaded successfully.


In [ ]:
def generate(model, context, max_new_tokens=50, temperature=1.0, top_k=None, top_p=None, repetition_penalty=1.0):
    return model.generate(
        context,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        repetition_penalty=repetition_penalty
    )

In [7]:
# 1. Decoding Strategies Test
test_prompt = "The future of AI is "
context = torch.tensor([encode(test_prompt)], dtype=torch.long, device=device)

print("Prompt:", test_prompt)

# Greedy Decoding (k=1)
out = generate(model, context, max_new_tokens=50, top_k=1, temperature=1.0)
print("\n1. Greedy Decoding:")
print(decode(out[0].tolist()))

# Temperature Sampling
out = generate(model, context, max_new_tokens=50, temperature=0.7)
print("\n2. High Temperature (1.5):")
print(decode(out[0].tolist()))

# Top-k Sampling
out = generate(model, context, max_new_tokens=50, temperature=0.8, top_k=5)
print("\n3. Top-k (k=5):")
print(decode(out[0].tolist()))

# Top-p (Nucleus) Sampling
out = generate(model, context, max_new_tokens=50, temperature=0.8, top_p=0.9)
print("\n4. Top-p (p=0.9):")
print(decode(out[0].tolist()))

Prompt: The future of AI is 

1. Greedy Decoding:
The future of AI is also known as the . = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =

2. High Temperature (1.5):
The future of AI is also created by the Republic of Ireland . However , the Republic of Ireland is unknown as a turn in May and his conservation of the Republic . = = = Irish and Irish Republic of English and culture = = = = = The Irish Republic of Ireland

3. Top-k (k=5):
The future of AI is not the best-based team . = = = = = = = = = The number of FITs are unchanged to FIFA-FIFA , which are also a FIFA Cup in 2010 , and has been announced that FIFA

4. Top-p (p=0.9):
The future of AI is often used in the United States , and the First World War II , and it was abandoned in 1942 . = = = History = = The CAT had been established in 1887 by the Great Britain . In 1877 , the area was renamed


In [ ]:
queries = [
    "Born in 1879, Albert Einstein was a ",                  # Biography
    "The American Civil War, which began in 1861, ",         # History
    "The city of Paris is located along the ",               # Geography
    "Photosynthesis is a process used by plants to ",        # Science/Biology
    "The Roman Empire was one of the largest in ",           # Ancient History
    "The Mona Lisa is a painting by the Italian artist ",    # Arts/Culture
    "The novel was published in 1925 and tells the story of ", # Literature
    "During the Second World War, the Allied forces ",       # Military History
    "The Earth's atmosphere is composed primarily of ",      # Earth Science
    "The company was founded in 1976 by Steve Jobs and "     # Business/Tech
]

print("\n--- 10 Different Queries (Top-p Sampling, temp=0.7, top_k=40, rep_penalty=1.2) ---")
for q in queries:
    ctx = torch.tensor([encode(q)], dtype=torch.long, device=device)
    out = generate(model, ctx, max_new_tokens=50, temperature=0.7, top_p=0.9, top_k=40, repetition_penalty=1.2)
    print(f"Query: {q}")
    print(f"Output: {decode(out[0].tolist())}\n")



--- 10 Different Queries (Top-p Sampling, temp=0.7, top_k=40, rep_penalty=1.2) ---
Query: Born in 1879, Albert Einstein was a 
Output: Born in 1879, Albert Einstein was a professional football manager . He then moved to New England with an early 1909 – 07 season after he joined his career at the end of the League Cup and McGregor . Woodhouse scored four goals in January 1881 , where he was

Query: The American Civil War, which began in 1861, 
Output: The American Civil War, which began in 1861, and that the War would be used to create a new government . In 1877 , this proximately 100 , the Mint-Patterson 's National Regular , was assigned to Mill Hall at St James . In 1882 , he

Query: The city of Paris is located along the 
Output: The city of Paris is located along the Bouston area . , a small-room window with two nave and two other cargo on its west wall , is a long-floor in the east end . This image is income to the north wall that has a large western

Query: Photosynthesis is a p

In [ ]:
# 3. Hallucination Check
hallucination_query = "The iPhone 25 Pro Max, released in 2035, features a "
ctx = torch.tensor([encode(hallucination_query)], dtype=torch.long, device=device)
out = generate(model, ctx, max_new_tokens=40, temperature=0.8, top_p=0.9)

print("\n--- Hallucination Test ---")
print(f"Query: {hallucination_query}")
print(f"Output: {decode(out[0].tolist())}")


--- Hallucination Test ---
Query: The iPhone 25 Pro Max, released in 2035, features a 
Output: The iPhone 25 Pro Max, released in 2035, features a full-@-@ class triple products from and then over 30,000 feet long-sight to their lower-down-of the time , averaged a day-to-white @-@
